# Avito Antibot: решение v3

Задача — оценить вероятность автоматизированного сбора данных для каждой `cookie_id` по событиям суточного окна. Основная метрика — максимальный Precision при Recall не ниже 0.7; куки с одинаковым score учитываются одной группой.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

PUBLIC_DIR = Path.cwd().resolve()
assert (PUBLIC_DIR / 'solution.ipynb').is_file()
ROOT = next((p for p in [PUBLIC_DIR, *PUBLIC_DIR.parents]
             if all((p / 'data' / name).is_file()
                    for name in ['train.csv', 'test.csv', 'events.csv.gz'])), None)
assert ROOT is not None, 'Распакуйте исходный архив: нужны data/train.csv, data/test.csv, data/events.csv.gz'


## Данные и временные окна

Три файла исходного архива находятся в `data/` рядом с `solution.ipynb` или в одном из родительских каталогов. Для каждой cookie используются только события с `window_start_ts ≤ event_ts < window_end_ts`; границы окна берутся из `train.csv` и `test.csv`. `target` участвует в обучении и оценке, но не в расчёте признаков.

In [2]:
train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])
assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert 'target' in train and 'target' not in test
assert train.target.notna().all() and set(train.target.unique()) == {0, 1}
assert train.window_start_ts.lt(train.window_end_ts).all()
assert test.window_start_ts.lt(test.window_end_ts).all()


## Поведенческие признаки

Первые 94 признака описывают частоты действий, разнообразие объявлений и поиска, временной ритм, курсор, платформу/User-Agent, возраст cookie и переходы между действиями. Новый `raw_duplicate_count` считает точные повторы строк событий внутри окна по исходным данным. Для остальных признаков дубли удаляются, чтобы повторная запись не искажала агрегаты. Неопределённые значения заменяются на `−1`, а счётчики отсутствующих действий равны `0`. Идентификаторы cookie и объявлений в модель не передаются.

In [3]:
EVENTS = ['item_view','search_results_view','photo_swipe','favorite_add','seller_page_view','contact_phone_show','login','contact_chat_open','contact_message_sent','captcha_shown']

def in_window(events, meta):
    m = meta[['cookie_id','cookie_created_at','window_start_ts','window_end_ts']]
    assert m.cookie_id.is_unique
    e = events.merge(m, on='cookie_id', validate='many_to_one')
    return e[e.event_ts.ge(e.window_start_ts) & e.event_ts.lt(e.window_end_ts)].copy()

def features(events, meta):
    e = in_window(events, meta).drop_duplicates().sort_values(['cookie_id','event_ts'])
    g = e.groupby('cookie_id', sort=False)
    f = pd.DataFrame(index=pd.Index(meta.cookie_id, name='cookie_id'))
    f['n_events'] = g.size()
    for name in EVENTS:
        counts = e.loc[e.event_name.eq(name)].groupby('cookie_id').size().reindex(f.index, fill_value=0)
        f['count_'+name] = counts
        f['share_'+name] = counts / f.n_events
    for col in ['item_id','item_category','item_location','search_query','seller_type']:
        f[col+'_unique'] = g[col].nunique()
        f[col+'_unique_ratio'] = g[col].nunique() / g[col].count().replace(0,np.nan)
    for stat in ['mean','std','max','median','nunique']:
        f['search_page_'+stat] = getattr(g.search_page,stat)()
    first, last = g.event_ts.min(), g.event_ts.max()
    f['activity_seconds'] = (last-first).dt.total_seconds()
    f['events_per_second'] = f.n_events/(f.activity_seconds+1)
    e['gap'] = g.event_ts.diff().dt.total_seconds()
    gaps = e.groupby('cookie_id').gap
    for stat in ['mean','std','min','max','median']:
        f['gap_'+stat] = getattr(gaps,stat)()
    f['gap_cv'] = f.gap_std/(f.gap_mean+1e-6)
    for key, mask in [('short',e.gap.le(1)),('long',e.gap.gt(300)),('zero',e.gap.eq(0))]:
        f['gap_'+key+'_share'] = mask.groupby(e.cookie_id).sum()/(f.n_events-1).clip(lower=1)
    e['hour'] = ((e.event_ts-e.window_start_ts).dt.total_seconds()//3600).astype(int)
    h = e.groupby(['cookie_id','hour']).size()
    f['active_hours'] = h.groupby(level=0).size()
    f['max_hour_share'] = h.groupby(level=0).max()/f.n_events
    hp = h / h.groupby(level=0).transform('sum')
    f['hour_entropy'] = (-(hp*np.log(hp))).groupby(level=0).sum()
    p = e.dropna(subset=['pointer_x','pointer_y']).copy()
    pg = p.groupby('cookie_id')
    f['pointer_share'] = pg.size()/f.n_events
    for col in ['pointer_x','pointer_y']:
        for stat in ['std','nunique']:
            f[col+'_'+stat] = getattr(pg[col],stat)()
    p['step'] = np.hypot(pg.pointer_x.diff(),pg.pointer_y.diff())
    for stat in ['mean','std','median']:
        f['pointer_step_'+stat] = getattr(p.groupby('cookie_id').step,stat)()
    f['pointer_repeat_share'] = p.step.eq(0).groupby(p.cookie_id).sum()/(pg.size()-1).clip(lower=1)
    e['platform_norm'] = e.platform.str.lower().replace({'desktop':'web','iphone':'ios'})
    f['platform_unique'] = e.groupby('cookie_id').platform_norm.nunique()
    for name in ['web','android','ios']:
        f['platform_'+name+'_share'] = e.platform_norm.eq(name).groupby(e.cookie_id).mean()
    ua = e.user_agent.fillna('').str.lower()
    for name,pattern in {'automation':'headless|selenium|python|curl|wget|bot|scrapy', 'mobile':'mobile|android|iphone', 'app':'^avito/', 'firefox':'firefox', 'chrome':'chrome', 'yandex':'yabrowser'}.items():
        f['ua_'+name+'_share'] = ua.str.contains(pattern,regex=True).groupby(e.cookie_id).mean()
    m = meta.set_index('cookie_id')
    f['cookie_age_days'] = (m.window_end_ts-m.cookie_created_at).dt.total_seconds()/86400
    assert not f.cookie_age_days.lt(0).any()
    f['no_events'] = f.n_events.isna().astype(int)
    return f.replace([np.inf,-np.inf],np.nan).fillna(-1).astype(float)

def extra_features(events, meta):
    """Независимые агрегаты событий каждой cookie строго внутри её окна."""
    e = in_window(events, meta[['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts']])
    e = e.drop_duplicates().sort_values(['cookie_id', 'event_ts'])
    g = e.groupby('cookie_id', sort=False)
    idx = pd.Index(meta.cookie_id, name='cookie_id')
    t = pd.DataFrame(index=idx)
    gap = g.event_ts.diff().dt.total_seconds()
    gg = gap.groupby(e.cookie_id)
    for q in [0.1, 0.25, 0.75, 0.9]:
        t[f'new_gap_q{int(q * 100)}'] = gg.quantile(q)
    t['new_gap_iqr'] = t.new_gap_q75 - t.new_gap_q25
    denom = (g.size() - 1).clip(lower=1)
    for name, mask in [
        ('1_to_5s', gap.gt(1) & gap.le(5)),
        ('5_to_60s', gap.gt(5) & gap.le(60)),
        ('over_15min', gap.gt(900)),
    ]:
        t[f'new_gap_{name}_share'] = mask.groupby(e.cookie_id).sum() / denom
    minute = ((e.event_ts - e.window_start_ts).dt.total_seconds() // 60).astype(int)
    per_minute = e.groupby([e.cookie_id, minute]).size()
    t['new_active_minutes'] = per_minute.groupby(level=0).size()
    t['new_max_events_per_minute'] = per_minute.groupby(level=0).max()
    t['new_top_minute_share'] = per_minute.groupby(level=0).max() / g.size()
    t['new_median_events_per_minute'] = per_minute.groupby(level=0).median()

    previous_action = g.event_name.shift()
    pairs = [
        ('item_to_item', 'item_view', 'item_view'),
        ('search_to_item', 'search_results_view', 'item_view'),
        ('item_to_search', 'item_view', 'search_results_view'),
        ('search_to_search', 'search_results_view', 'search_results_view'),
        ('item_to_photo', 'item_view', 'photo_swipe'),
    ]
    for name, before, after in pairs:
        flag = previous_action.eq(before) & e.event_name.eq(after)
        t[f'new_transition_{name}_share'] = flag.groupby(e.cookie_id).sum() / denom
    item_views = e[e.event_name.eq('item_view') & e.item_id.notna()]
    iv = item_views.groupby('cookie_id').item_id
    t['new_item_view_revisit_share'] = 1 - iv.nunique() / iv.size()
    item_previous = g.item_id.shift()
    consecutive_item = e.item_id.notna() & e.item_id.eq(item_previous)
    t['new_same_item_adjacent_share'] = consecutive_item.groupby(e.cookie_id).sum() / denom
    for col in ['item_category', 'item_location', 'search_query']:
        nonmissing = e[e[col].notna()].groupby('cookie_id')[col]
        top_count = nonmissing.value_counts().groupby(level=0).max()
        t[f'new_{col}_top_share'] = top_count / nonmissing.size()
    search = e[e.search_page.notna()]
    sg = search.groupby('cookie_id').search_page
    t['new_deep_search_share'] = search.search_page.gt(5).groupby(search.cookie_id).sum() / sg.size()
    return t.replace([np.inf, -np.inf], np.nan).fillna(-1).astype(float)

def raw_duplicate_feature(events, meta):
    # Повторы считаются по сырым событиям внутри окна.
    e = in_window(events, meta)
    idx = pd.Index(meta.cookie_id, name='cookie_id')
    counts = e.duplicated().groupby(e.cookie_id).sum().reindex(idx, fill_value=0)
    return pd.DataFrame({'raw_duplicate_count': counts.astype(float)}, index=idx)


In [4]:
metadata = pd.concat([train.drop(columns='target'), test], ignore_index=True)
observed = in_window(events, metadata)
assert observed.event_ts.ge(observed.window_start_ts).all()
assert observed.event_ts.lt(observed.window_end_ts).all()
base = features(events, metadata)
extra = extra_features(events, metadata)
duplicates = raw_duplicate_feature(events, metadata)
assert base.shape == (len(metadata), 71) and extra.shape == (len(metadata), 23) and duplicates.shape == (len(metadata), 1)
assert base.index.equals(extra.index)
# Проверка: события вне окна не влияют на признаки.
pd.testing.assert_frame_equal(base, features(observed[events.columns], metadata))
pd.testing.assert_frame_equal(extra, extra_features(observed[events.columns], metadata))
pd.testing.assert_frame_equal(duplicates, raw_duplicate_feature(observed[events.columns], metadata))
all_features = pd.concat([base, extra, duplicates], axis=1)
assert np.isfinite(all_features.to_numpy()).all()
X_train = all_features.loc[train.cookie_id]
X_test = all_features.loc[test.cookie_id]
assert X_train.columns.equals(X_test.columns)
assert X_train.index.equals(pd.Index(train.cookie_id, name='cookie_id'))
assert X_test.index.equals(pd.Index(test.cookie_id, name='cookie_id'))


## Модель

`CatBoostClassifier` обучается с 500 итерациями, глубиной 6, шагом 0.05, seed 0 и четырьмя потоками. В предыдущих экспериментах сравнивались RandomForest и CatBoost на тех же временных разбиениях. CatBoost с поведенческими признаками показал более высокую P@R≥0.7; параметры финальной модели совпадают с валидацией.

In [5]:
CATBOOST_PARAMS = dict(iterations=500, depth=6, learning_rate=0.05,
                       loss_function='Logloss', random_seed=0,
                       thread_count=4, verbose=False, allow_writing_files=False)


## Временная валидация

Основной срез: обучение на окнах до 17 апреля 2026 года, проверка 17–19 апреля. Дополнительный: обучение до 14 апреля, проверка 14–16 апреля. Окна обучающих cookie заканчиваются не позже начала соответствующей проверки. Код ниже сравнивает quickstart-baseline на основном срезе и модели v2/v3 на обоих; P@R≥0.7 вычисляется функцией из официального `metric.py`, AP служит дополнительной метрикой.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# Функция из официального metric.py учитывает одинаковые score вместе.
TARGET_RECALL = 0.70

def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # Последняя строка каждой группы равных score.
    return tp[ends] / k[ends], tp[ends] / n_pos

def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0

# Базовая модель quickstart: два агрегата по исходным событиям.
raw_train_events = in_window(events, train.drop(columns='target'))
basic = pd.DataFrame(index=pd.Index(train.cookie_id, name='cookie_id'))
basic['n_events'] = raw_train_events.groupby('cookie_id').size()
basic['item_nunique'] = raw_train_events.groupby('cookie_id').item_id.nunique()
basic = basic.fillna(0)

rows = []
for split, cut, end in [('main', '2026-04-17', '2026-04-20'),
                        ('earlier', '2026-04-14', '2026-04-17')]:
    fit = train.window_start_ts.lt(cut).to_numpy()
    valid = train.window_start_ts.ge(cut).to_numpy() & train.window_start_ts.lt(end).to_numpy()
    assert train.loc[fit, 'window_end_ts'].le(pd.Timestamp(cut)).all()
    fit_ids = train.loc[fit, 'cookie_id']
    valid_ids = train.loc[valid, 'cookie_id']
    y_fit = train.loc[fit, 'target'].to_numpy()
    y_valid = train.loc[valid, 'target'].to_numpy()
    if split == 'main':
        baseline_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                random_state=0)
        baseline_model.fit(basic.loc[fit_ids], y_fit)
        baseline_score = baseline_model.predict_proba(basic.loc[valid_ids])[:, 1]
        rows.append((split, 'quickstart RF', precision_at_recall(y_valid, baseline_score),
                     average_precision_score(y_valid, baseline_score)))
    for name, columns in [('v2', X_train.columns[:94]), ('v3', X_train.columns)]:
        validation_model = CatBoostClassifier(**CATBOOST_PARAMS)
        validation_model.fit(X_train.loc[fit_ids, columns], y_fit)
        score_valid = validation_model.predict_proba(X_train.loc[valid_ids, columns])[:, 1]
        rows.append((split, name, precision_at_recall(y_valid, score_valid),
                     average_precision_score(y_valid, score_valid)))
validation_results = pd.DataFrame(rows, columns=['period', 'model', 'P@R≥0.7', 'AP'])
display(validation_results.round(6))


,period,model,P@R≥0.7,AP
0,main,quickstart RF,0.102470,0.174613
1,main,v2,0.751678,0.783006
2,main,v3,0.767123,0.783065
3,earlier,v2,0.713542,0.776753
4,earlier,v3,0.721053,0.777930


## Эксперименты и выбор v3

Quickstart с двумя признаками дал P@R≥0.7 **0.10247** на основном срезе. Ансамбль RandomForest+CatBoost v1 получил **0.69939 / 0.69192**, CatBoost v2 — **0.75168 / 0.71354** на основном и дополнительном срезах. Проверки последовательностей действий, возвратов к объявлениям, сокращения признаков и выхода из поиска не дали одновременного прироста P@R≥0.7 и AP на обоих срезах. Один признак дублей повысил P@R≥0.7 до **0.76712 / 0.72105**; AP составила **0.78306 / 0.77793** против **0.78301 / 0.77675** у v2. На Stepik v3 получила **0.79557**, v2 — **0.79361**. Разница невелика.

## Итог и воспроизведение

Последняя ячейка обучает модель на всём `train.csv` и сохраняет `submission.csv` с колонками `cookie_id,score`. Проверенное окружение — Python **3.14.5** и версии библиотек из `requirements.txt`; исходные файлы размещаются в `data/`, ячейки выполняются по порядку из каталога с notebook. NumPy, pandas, scikit-learn и CatBoost работают локально, внешние API не используются. Оба временных среза многократно участвовали в экспериментах; частота дублей может зависеть от способа записи событий, поэтому небольшой прирост не гарантирует такого же результата на другом потоке.

In [7]:
model = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                           loss_function='Logloss', random_seed=0,
                           thread_count=4, verbose=False, allow_writing_files=False)
model.fit(X_train, train.target.to_numpy())
score = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({'cookie_id': test.cookie_id, 'score': score})
assert list(submission.columns) == ['cookie_id', 'score']
assert submission.cookie_id.is_unique and submission.cookie_id.equals(test.cookie_id)
assert not submission.isna().any().any() and submission.score.between(0, 1).all()
output = PUBLIC_DIR / 'submission.csv'
submission.to_csv(output, index=False)
saved = pd.read_csv(output, float_precision='round_trip')
assert saved.cookie_id.equals(test.cookie_id)
assert np.array_equal(saved.score.to_numpy(), score)
print(f'Сохранено: {output.name}, {len(saved)} строк')


Сохранено: submission.csv, 4909 строк
